# Notebook 4 — From Qiskit to IBM Quantum Hardware

**Qiskit Fall Fest 2026 — University of Ottawa**

**Difficulty:** Intermediate
**Estimated time:** 100–120 minutes
**Prerequisites:** Notebooks 1–3 (circuits, measurement, observables, Aer simulation and noise).

## Learning objectives
By the end of this notebook you will be able to:

- ✓ authenticate to IBM Quantum securely (without ever hard-coding a token)
- ✓ list and select real IBM Quantum backends
- ✓ visualize hardware connectivity (coupling maps) and circuit layouts
- ✓ explain why transpilation is required and use `generate_preset_pass_manager`
- ✓ run circuits on real hardware using `SamplerV2` and `EstimatorV2`
- ✓ retrieve, save, and interpret hardware results, including job IDs
- ✓ compare ideal, noisy-simulated, and real-hardware results in one picture

> **⚠️ Hardware cells.** Cells that require an active IBM Quantum account and real credentials are clearly marked **`# HARDWARE CELL`** at the top. Every other cell in this notebook runs successfully without any credentials — you can read and understand the entire workflow even before you have (or without ever needing) an IBM Quantum API token.

## 1. The IBM Quantum Platform

$$\text{Create} \to \text{Simulate} \to \text{Select hardware} \to \text{Transpile} \to \text{Run} \to \text{Retrieve} \to \text{Analyze} \to \text{Compare}$$

Key vocabulary:

- **QPU** — Quantum Processing Unit, the physical chip containing the qubits.
- **Backend** — a specific QPU (or simulator) you submit jobs to.
- **Job** — one submitted unit of work (one or more circuits) sent to a backend.
- **Queue** — real hardware is shared among many users; your job waits its turn.
- **Shots** — number of repeated circuit executions per job, as in Notebooks 2–3.
- **Runtime** — IBM's service layer (`qiskit-ibm-runtime`) that manages authentication, backend selection, and primitive execution.
- **Credentials** — your personal IBM Quantum API token, used to authenticate `QiskitRuntimeService`.

This notebook follows IBM's modern four-stage pattern for hardware execution: **Map** (build the circuit) → **Optimize** (transpile) → **Execute** (run via primitives) → **Analyze** (interpret results).

## 2. Authentication

> **🔒 Never commit an IBM Quantum API token to GitHub or paste it directly into a notebook cell.** Anyone with your token can submit jobs (and consume your quota) on your account.

The recommended workflow is to **save your credentials once, locally**, and then load them by reference in every notebook — never as a literal string in code you might commit or share.

```python
# Run this ONCE, in a private cell you do NOT commit to version control:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token="<PASTE_YOUR_TOKEN_HERE_ONLY_LOCALLY_NEVER_COMMITTED>",
    channel="ibm_quantum_platform",   # current IBM Quantum channel identifier — verify against your account type
    overwrite=True,
)
```

After running that once, every future notebook can simply do:

```python
service = QiskitRuntimeService()   # loads saved credentials automatically, no token in this notebook
```

If you're using a shared or hackathon-provided machine, consider using an environment variable or a `.env` file (excluded via `.gitignore`) instead of `save_account`, so credentials never touch disk in a way that could be accidentally committed.

## 3. QiskitRuntimeService

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

try:
    service = QiskitRuntimeService()
    HAVE_CREDENTIALS = True
    print("Connected to IBM Quantum Runtime successfully.")
except Exception as e:
    HAVE_CREDENTIALS = False
    print("No IBM Quantum credentials found — hardware cells will be skipped in this run.")
    print("(This is expected if you have not yet run QiskitRuntimeService.save_account().)")

`QiskitRuntimeService` is your entry point to everything hardware-related: listing backends, submitting jobs, and retrieving results. We wrap the connection in a `try/except` and store a `HAVE_CREDENTIALS` flag so that **the rest of this notebook can run end-to-end even without an account** — hardware-only cells below check this flag before attempting to submit real jobs.

## 4. Explore Available Backends

In [ ]:
# HARDWARE CELL — requires IBM Quantum credentials
if HAVE_CREDENTIALS:
    backends = service.backends()
    for b in backends:
        status = b.status()
        print(f"{b.name:25s}  qubits={b.num_qubits:<4}  operational={status.operational}  pending_jobs={status.pending_jobs}")
else:
    print("Skipped: no credentials. Example output would look like:")
    print("ibm_sherbrooke            qubits=127   operational=True   pending_jobs=42")

In [ ]:
# HARDWARE CELL — requires IBM Quantum credentials
if HAVE_CREDENTIALS:
    backend = service.least_busy(operational=True, min_num_qubits=5)
    print("Selected backend:", backend.name)
else:
    backend = None
    print("Skipped: no credentials. `service.least_busy(...)` returns the operational backend with the shortest queue.")

> **Important.** `least_busy()` picks the backend with the *shortest current queue* among those matching your filters — it does **not** consider error rates, topology, or whether that backend is well-suited to your specific circuit. For serious work, you should also inspect a backend's `num_qubits`, `operational` status, `target.basis_gates` (its native gate set), and coupling map (Section 5) before committing to it.

## 5. Hardware Connectivity Visualization

Real qubits are **not all connected to each other** — each qubit can typically only interact directly with a handful of physical neighbors. This is the **coupling map**, and it's one of the most important constraints transpilation has to work around.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.visualization import plot_gate_map

fake_backend = FakeSherbrooke()   # always available, no credentials needed — used for all layout/connectivity demos below
plot_gate_map(fake_backend)

We use `FakeSherbrooke` here (from Notebook 3) specifically so that **every visualization and transpilation demo in this section works without any credentials** — the same functions work identically on a real `backend` object if you have one connected.

## 6. Why Transpilation Exists

$$\text{Logical circuit} \xrightarrow{\text{transpilation}} \text{Hardware-compatible (ISA) circuit}$$

Your logical circuit (built with high-level gates like `cx`, `h`, `ry`) usually cannot run directly on hardware, because:

- **Basis gates** — hardware only implements a small native gate set (e.g. `rz`, `sx`, `x`, `cx` or `ecr`); other gates must be decomposed into these.
- **Connectivity** — a two-qubit gate can only be applied directly between physically connected qubits; gates between distant qubits require inserted **SWAP** operations to bring them together first.
- **Optimization** — redundant or cancelable gates can often be removed to reduce circuit depth (and therefore noise).

`generate_preset_pass_manager` builds the entire transpilation pipeline for you, tuned to a specific backend's basis gates and coupling map.

In [ ]:
from qiskit import QuantumCircuit
# Qiskit exposes this helper at different import paths across versions.
# Newer Qiskit releases support the shorter qiskit.transpiler import;
# Qiskit 1.x uses qiskit.transpiler.preset_passmanagers.
try:
    from qiskit.transpiler import generate_preset_pass_manager
except ImportError:
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

pm = generate_preset_pass_manager(optimization_level=1, backend=fake_backend)
isa_bell = pm.run(bell)
print("Transpiled successfully using FakeSherbrooke's basis gates and coupling map.")

## 7. Compare Original and Transpiled Circuits

In [ ]:
print("Original  -> depth:", bell.depth(), " size:", bell.size(), " ops:", bell.count_ops())
print("Transpiled-> depth:", isa_bell.depth(), " size:", isa_bell.size(), " ops:", isa_bell.count_ops())

In [ ]:
bell.draw("mpl")

In [ ]:
isa_bell.draw("mpl", idle_wires=False)

> **Checkpoint — why did the circuit become larger?** The original `h`/`cx` gates aren't in `FakeSherbrooke`'s native basis gate set, so each had to be decomposed into multiple native gates (typically several `rz`/`sx` rotations plus a native two-qubit gate like `ecr`). More instructions generally means more accumulated noise on real hardware — this is exactly why minimizing transpiled depth matters.

## 8. Layout

**Layout** is the mapping from your circuit's *logical* qubits (indices 0, 1, 2, ... as you wrote them) to the backend's *physical* qubits (specific locations on the chip). The transpiler chooses this mapping to minimize the number of SWAPs needed, based on the coupling map.

In [ ]:
from qiskit.visualization import plot_circuit_layout

plot_circuit_layout(isa_bell, fake_backend)

This plot highlights exactly *which* physical qubits on the chip your two logical qubits ended up mapped to — useful for sanity-checking that the transpiler picked well-connected, low-error physical qubits.

## 9. SamplerV2 on Hardware

In [ ]:
bell_meas = QuantumCircuit(2, 2)
bell_meas.h(0)
bell_meas.cx(0, 1)
bell_meas.measure([0, 1], [0, 1])

pm_meas = generate_preset_pass_manager(optimization_level=1, backend=fake_backend)
isa_bell_meas = pm_meas.run(bell_meas)
print("Circuit ready for SamplerV2. depth:", isa_bell_meas.depth())

In [ ]:
# HARDWARE CELL — requires IBM Quantum credentials
from qiskit_ibm_runtime import SamplerV2

if HAVE_CREDENTIALS:
    pm_real = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_bell_real = pm_real.run(bell_meas)

    sampler = SamplerV2(mode=backend)
    job = sampler.run([isa_bell_real], shots=1024)
    print("Job submitted. Job ID:", job.job_id())

    result = job.result()
    hw_counts = result[0].data.c.get_counts()
    print("Hardware counts:", hw_counts)
else:
    print("Skipped: no credentials.")
    print("Expected result shape: result[0].data.c.get_counts() -> a dict like {'00': 480, '11': 470, '01': 40, '10': 34}")
    hw_counts = None
    job = None

**Retrieving results with `SamplerV2`.** The result object is indexed by PUB (the order you passed circuits in), and `.data.<creg_name>.get_counts()` gives you the familiar counts dictionary — the classical register name (here `c`) must match the name you gave it (or `meas` if you used `measure_all()`), same as in Notebooks 2–3.

## 10. EstimatorV2 on Hardware

In [ ]:
from qiskit.quantum_info import SparsePauliOp

bell_no_meas = QuantumCircuit(2)
bell_no_meas.h(0)
bell_no_meas.cx(0, 1)

pm_est = generate_preset_pass_manager(optimization_level=1, backend=fake_backend)
isa_bell_no_meas = pm_est.run(bell_no_meas)

observables = [SparsePauliOp("ZI"), SparsePauliOp("IZ"), SparsePauliOp("ZZ"), SparsePauliOp("XX")]
# Observables must be re-mapped ("applied to the transpiled layout") to match physical qubit positions:
mapped_observables = [obs.apply_layout(isa_bell_no_meas.layout) for obs in observables]
print("Observables re-mapped to transpiled physical layout.")

> **Why does this matter?** After transpilation, your circuit's logical qubits are mapped onto *specific physical qubits* that may not match their original indices. If you don't call `.apply_layout(...)` on your observables, `EstimatorV2` would compute expectation values for the wrong physical qubits — a very common source of silently-wrong hardware results.

In [ ]:
# HARDWARE CELL — requires IBM Quantum credentials
from qiskit_ibm_runtime import EstimatorV2

if HAVE_CREDENTIALS:
    pm_est_real = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_bell_est_real = pm_est_real.run(bell_no_meas)
    mapped_obs_real = [obs.apply_layout(isa_bell_est_real.layout) for obs in observables]

    estimator = EstimatorV2(mode=backend)
    job_est = estimator.run([(isa_bell_est_real, mapped_obs_real)])
    print("Job submitted. Job ID:", job_est.job_id())

    result_est = job_est.result()
    hw_evs = result_est[0].data.evs
    labels = ["ZI", "IZ", "ZZ", "XX"]
    for label, val in zip(labels, hw_evs):
        print(f"<{label}> = {val:+.3f}")
else:
    print("Skipped: no credentials.")
    print("Expected shape: result_est[0].data.evs -> array of 4 expectation values, one per observable.")
    hw_evs = None

## 11. Compare Ideal, Noisy, and Real Hardware

This is the central experiment of this notebook: run the *same* Bell-state circuit through three different execution paths and compare.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit.primitives import StatevectorSampler

# Path 1: Ideal simulation
ideal_sampler = StatevectorSampler()
ideal_counts = ideal_sampler.run([bell_meas], shots=4000).result()[0].data.c.get_counts()

# Path 2: Aer noisy simulation using FakeSherbrooke's realistic noise model
noise_model = NoiseModel.from_backend(fake_backend)
sim_noisy = AerSimulator(noise_model=noise_model)
pm_sim_noisy = generate_preset_pass_manager(optimization_level=1, backend=sim_noisy)
noisy_counts = sim_noisy.run(pm_sim_noisy.run(bell_meas), shots=4000).result().get_counts()

print("Ideal simulation counts:", ideal_counts)
print("Aer noisy simulation counts:", noisy_counts)
print("Hardware counts:", hw_counts if hw_counts else "(skipped — no credentials)")

In [ ]:
from qiskit.visualization import plot_histogram

all_results = {"Ideal": ideal_counts, "Aer noisy": noisy_counts}
if hw_counts:
    all_results["Real hardware"] = hw_counts

plot_histogram(list(all_results.values()), legend=list(all_results.keys()))

**Interpretation.** Ideal simulation gives a clean 50/50 `00`/`11` split. Aer noisy simulation (using a realistic fake-backend noise model) shows some leakage into `01`/`10` from simulated gate and readout errors. Real hardware (when available) typically shows a *similar pattern* of leakage — confirming that our noise model from Notebook 3 is a reasonably good predictor of real device behaviour, though real hardware will also have effects (crosstalk, drift, calibration changes) that a static noise-model snapshot can't fully capture.

## 12. Hardware Results Are Data — Save Them

Hardware jobs cost queue time and (on non-free plans) money — **never treat a completed hardware job as disposable.** Save everything you might need later in ordinary Python structures.

In [ ]:
import json
from datetime import datetime, timezone

# HARDWARE CELL — requires IBM Quantum credentials
if HAVE_CREDENTIALS and job is not None:
    saved_result = {
        "job_id": job.job_id(),
        "backend_name": backend.name,
        "shots": 1024,
        "counts": hw_counts,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    with open("bell_hardware_result.json", "w") as f:
        json.dump(saved_result, f, indent=2)
    print("Saved bell_hardware_result.json")
else:
    print("Skipped: no credentials / no job. Example structure that WOULD be saved:")
    print(json.dumps({
        "job_id": "<job-id-string>",
        "backend_name": "<backend-name>",
        "shots": 1024,
        "counts": {"00": 480, "11": 470, "01": 40, "10": 34},
        "timestamp": "<ISO-8601 timestamp>",
    }, indent=2))

## 13. Job IDs

> **Important.** Always save `job.job_id()` immediately after submission. If your notebook crashes, your laptop sleeps, or you close your session while a job is still queued, the job **keeps running on IBM's servers** — you just need the job ID to retrieve it later, without resubmitting (and re-queueing) the same circuit.

```python
# Retrieving a previously submitted job later, using only its ID:
# job = service.job("<job_id_you_saved_earlier>")
# result = job.result()
```

## 14. Practical Hardware Advice

- Always **simulate first** (ideally with a realistic fake-backend noise model) before spending real queue time.
- **Test small circuits** — verify correctness on 1–2 qubits before scaling up.
- **Always transpile** for your specific target backend before submitting — an untranspiled circuit will typically error out entirely.
- **Inspect `depth()`** of your transpiled circuit — deep circuits accumulate more noise and are more likely to give unreliable results.
- **Avoid unnecessarily deep circuits** — look for redundant gates, or try a higher `optimization_level` in `generate_preset_pass_manager`.
- **Save job IDs and results immediately** after submission and after retrieval, respectively.
- **Real hardware results will differ from ideal simulation** — this is expected, not a bug in your circuit.
- **Queue time can be significant**, especially on shared/free hardware access tiers — plan your hackathon time accordingly.
- **Do not rerun a job unnecessarily** — if a job is still processing or already completed, retrieve it by ID instead of resubmitting.

## 15. Final Challenge — Bell State: Simulator to QPU

This is your **graduation exercise** from the introductory Qiskit workshop. Working independently (referring back to Notebooks 1–4 as needed), complete the following pipeline:

1. Create a Bell-state circuit.
2. Visualize it with `qc.draw("mpl")`.
3. Simulate it ideally (`StatevectorSampler` or `Statevector`).
4. Measure $\langle ZZ\rangle$ and $\langle XX\rangle$ using `StatevectorEstimator`.
5. Transpile it for a real (or fake, if no credentials) backend using `generate_preset_pass_manager`.
6. Inspect the transpiled circuit's depth, size, and `count_ops()`.
7. If you have IBM Quantum credentials: run it on real hardware using `SamplerV2`. If not: run it on `FakeSherbrooke` with `NoiseModel.from_backend(...)` via `AerSimulator`.
8. Retrieve and visualize the results with `plot_histogram`.
9. Compare hardware/noisy results with your ideal simulation from step 3.
10. In a short markdown cell, **explain the discrepancy** you observe — which specific noise mechanisms from Notebook 3 most plausibly explain what you see?

Use the empty cells below to complete each step.

In [ ]:
# Step 1


In [ ]:
# Step 2


In [ ]:
# Step 3


In [ ]:
# Step 4


In [ ]:
# Step 5


In [ ]:
# Step 6


In [ ]:
# Step 7


In [ ]:
# Step 8


In [ ]:
# Step 9


In [ ]:
# Step 10


## Qiskit Cheat Sheet — Notebook 4

| Task | Code |
|---|---|
| Connect to IBM Quantum | `QiskitRuntimeService()` |
| List backends | `service.backends()` |
| Pick least-busy backend | `service.least_busy(operational=True, min_num_qubits=n)` |
| Coupling map plot | `plot_gate_map(backend)` |
| Transpile for hardware | `generate_preset_pass_manager(optimization_level, backend=backend).run(qc)` |
| Layout plot | `plot_circuit_layout(isa_qc, backend)` |
| Sample on hardware | `SamplerV2(mode=backend).run([isa_qc], shots=n)` |
| Get hardware counts | `result[0].data.<creg>.get_counts()` |
| Map observable to layout | `observable.apply_layout(isa_qc.layout)` |
| Estimate on hardware | `EstimatorV2(mode=backend).run([(isa_qc, mapped_observables)])` |
| Get expectation values | `result[0].data.evs` |
| Job ID | `job.job_id()` |
| Retrieve a saved job | `service.job(job_id)` |

## Common Mistakes

- **Trying to obtain a `Statevector` from hardware.** Real hardware only ever returns measurement outcomes — never the exact quantum state. That's a simulator-only tool (Notebooks 1–3).
- **Forgetting to transpile for the specific backend.** A logical circuit with non-native gates or disallowed qubit connections will fail on real hardware without transpilation.
- **Forgetting to map observables to the transpiled layout.** `EstimatorV2` needs `observable.apply_layout(isa_qc.layout)`, or it will silently evaluate the wrong physical qubits (Section 10).
- **Confusing qubit order in returned bitstrings.** The little-endian convention from Notebook 1 still applies to hardware results.
- **Hard-coding an API token in a notebook cell.** Always use `QiskitRuntimeService.save_account(...)` once, locally, and never commit the token.
- **Re-submitting a job you already ran.** Save `job.job_id()` and use `service.job(job_id)` to retrieve results instead of resubmitting and re-queueing.
- **Using deprecated APIs from old tutorials**, such as `backend.run(circuit)` directly, `IBMQ.load_account()`, `execute(...)`, or V1 `Sampler`/`Estimator` — this notebook consistently uses `QiskitRuntimeService`, `SamplerV2`, `EstimatorV2`, and `generate_preset_pass_manager` throughout.